# 00 Probe

Spec sections 13-14. One agent per model tier writes a bot (author call) and revises it once, with no teaching; a second independent agent of the same tier does the same; we cross-play the two.

**Scope: stub backend only.** No API key was used. Every number below exercises the pipeline (scores of perturbed anchor bots, fake token counts priced at the tier's list price) and says nothing about a real model. The same notebook runs unchanged once a real backend exists.

In [1]:
import json, os, sys
sys.path.insert(0, 'src')  # repo root is the working directory
os.environ.setdefault('PYTHONHASHSEED', '0')
from culture.analysis.probe import run_probe
OUT = 'runs/probe'
results = {m: run_probe('configs/probe.yaml', OUT, m) for m in ['claude-haiku-4-5', 'claude-sonnet-5']}

In [2]:
for m, r in results.items():
    print(m)
    for a, v in r['agents'].items():
        print(f"  {a}: self-play {v['selfplay']:.2f}, illegal {v['illegal_rate']:.3f}, {v['lines']} lines, H-group keywords {v['hgroup_keywords']}")
    print(f"  cross-play between the two independent agents: {r['crossplay_between_independent_agents']:.2f}")
    print(f"  conventions bag-of-words cosine between agents: {r['conventions_similarity_between_agents']:.2f}")
    for t, c in r['cost_per_call'].items():
        print(f"  {t}: {c['calls']} calls, {c['tokens_per_call']:.0f} tokens/call, ${c['usd_per_call']:.4f}/call (nominal, stub)")
    print('  gates:', r['gates'])

claude-haiku-4-5
  g0a0: self-play 16.93, illegal 0.000, 220 lines, H-group keywords {}
  g1a0: self-play 15.29, illegal 0.000, 220 lines, H-group keywords {}
  cross-play between the two independent agents: 15.91
  conventions bag-of-words cosine between agents: 0.98
  author: 2 calls, 3876 tokens/call, $0.0148/call (nominal, stub)
  revise: 2 calls, 8892 tokens/call, $0.0190/call (nominal, stub)
  teach: 1 calls, 3962 tokens/call, $0.0041/call (nominal, stub)
  gates: {'headroom_selfplay_below_20': True, 'cost_measured': True, 'null_alive_crossplay_well_below_selfplay': False}
claude-sonnet-5
  g0a0: self-play 6.53, illegal 0.000, 220 lines, H-group keywords {}
  g1a0: self-play 15.56, illegal 0.000, 220 lines, H-group keywords {}
  cross-play between the two independent agents: 9.12
  conventions bag-of-words cosine between agents: 0.94
  author: 2 calls, 3758 tokens/call, $0.0283/call (nominal, stub)
  revise: 2 calls, 8750 tokens/call, $0.0369/call (nominal, stub)
  teach: 1 calls

## Gates (decided before running, spec section 14)

- **Headroom**: generation-one bots below 20 in self-play.
- **Cost**: tokens per revise and teach call measured; stage sizes derive from them.
- **Null alive**: independent agents' cross-play well below their self-play (here: below 0.8 of mean self-play).

With the stub, the null gate is expected to **fail**: stub bots are all variants of one rule-list template and share card-index semantics, so they are mutually compatible. That is a property of the stub, not evidence about LLM-written conventions.

In [3]:
print(json.dumps({m: r['gates'] for m, r in results.items()}, indent=1))

{
 "claude-haiku-4-5": {
  "headroom_selfplay_below_20": true,
  "cost_measured": true,
  "null_alive_crossplay_well_below_selfplay": false
 },
 "claude-sonnet-5": {
  "headroom_selfplay_below_20": true,
  "cost_measured": true,
  "null_alive_crossplay_well_below_selfplay": false
 }
}
